# EconoCausal: Dynamic Pricing Using Double Machine Learning

## Project Objective
To investigate how product price affects the number of units sold using Double Machine Learning (DML).

## Research Question
How does product price influence product sales?

## Methodology
- Dataset preparation and validation
- Treatment and outcome selection
- Confounder assessment
- Double Machine Learning estimation
- Effect interpretation and limitations

## Variables
- Treatment: Price
- Outcome: Units Sold
- Potential controls: To be finalized after checking variable timing and causal relevance.

**Important:** Predictive relationships do not automatically establish causal effects.

In [9]:
# Import required libraries

import pandas as pd
import numpy as np
from pathlib import Path

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

print("Libraries imported successfully!")
print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)

Libraries imported successfully!
Pandas version: 3.0.6
NumPy version: 2.5.4


## 1. Load the Prepared Dataset

In this section, we load the cleaned dataset and inspect its structure, columns, and sample records before selecting causal variables.

In [10]:
# Locate the processed dataset

possible_paths = [
    Path("data/processed/causal_data.csv"),
    Path("../../data/processed/causal_data.csv")
]

DATA_PATH = next(
    (path.resolve() for path in possible_paths if path.exists()),
    None
)

if DATA_PATH is None:
    raise FileNotFoundError(
        "causal_data.csv was not found. Check the data/processed folder."
    )

# Load dataset
df = pd.read_csv(DATA_PATH, parse_dates=["Date"])

print("Dataset loaded successfully!")
print("File location:", DATA_PATH)
print("Dataset shape:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst five records:")
display(df.head())

Dataset loaded successfully!
File location: C:\Users\methu\Desktop\EconoCausal\data\processed\causal_data.csv
Dataset shape: (76000, 18)

Column names:
['Date', 'Store ID', 'Product ID', 'Category', 'Region', 'Inventory Level', 'Units Sold', 'Units Ordered', 'Price', 'Discount', 'Weather Condition', 'Promotion', 'Competitor Pricing', 'Seasonality', 'Epidemic', 'Demand', 'Month', 'DayOfWeek']

First five records:


,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Price,Discount,Weather Condition,Promotion,Competitor Pricing,Seasonality,Epidemic,Demand,Month,DayOfWeek
0,2022-01-01,S001,P0001,Electronics,North,195,102,252,72.72,5,Snowy,0,85.73,Winter,0,115,1,5
1,2022-01-01,S001,P0002,Clothing,North,117,117,249,80.16,15,Snowy,1,92.02,Winter,0,229,1,5
2,2022-01-01,S001,P0003,Clothing,North,247,114,612,62.94,10,Snowy,1,60.08,Winter,0,157,1,5
3,2022-01-01,S001,P0004,Electronics,North,139,45,102,87.63,10,Snowy,0,85.19,Winter,0,52,1,5
4,2022-01-01,S001,P0005,Groceries,North,152,65,271,54.41,0,Snowy,0,51.63,Winter,0,59,1,5


In [11]:
# Dataset structure
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

print("\nComplete column list:")
for number, column in enumerate(df.columns, start=1):
    print(f"{number}. {column}")

print("\nData types:")
display(df.dtypes.to_frame(name="Data Type"))

print("\nMissing values:")
display(df.isnull().sum().to_frame(name="Missing Values"))

Number of rows: 76000
Number of columns: 18

Complete column list:
1. Date
2. Store ID
3. Product ID
4. Category
5. Region
6. Inventory Level
7. Units Sold
8. Units Ordered
9. Price
10. Discount
11. Weather Condition
12. Promotion
13. Competitor Pricing
14. Seasonality
15. Epidemic
16. Demand
17. Month
18. DayOfWeek

Data types:


,Data Type
Date,datetime64[us]
Store ID,str
Product ID,str
Category,str
Region,str
Inventory Level,int64
Units Sold,int64
Units Ordered,int64
Price,float64
Discount,int64



Missing values:


,Missing Values
Date,0
Store ID,0
Product ID,0
Category,0
Region,0
Inventory Level,0
Units Sold,0
Units Ordered,0
Price,0
Discount,0


In [12]:
print("ALL 18 COLUMNS:\n")

for i, col in enumerate(df.columns, start=1):
    print(f"{i}. {col}")

print("\nMISSING VALUES:")
print(df.isnull().sum().to_string())

ALL 18 COLUMNS:

1. Date
2. Store ID
3. Product ID
4. Category
5. Region
6. Inventory Level
7. Units Sold
8. Units Ordered
9. Price
10. Discount
11. Weather Condition
12. Promotion
13. Competitor Pricing
14. Seasonality
15. Epidemic
16. Demand
17. Month
18. DayOfWeek

MISSING VALUES:
Date                  0
Store ID              0
Product ID            0
Category              0
Region                0
Inventory Level       0
Units Sold            0
Units Ordered         0
Price                 0
Discount              0
Weather Condition     0
Promotion             0
Competitor Pricing    0
Seasonality           0
Epidemic              0
Demand                0
Month                 0
DayOfWeek             0


In [13]:
print("Total missing values:", df.isnull().sum().sum())
print("Treatment column exists:", "Price" in df.columns)
print("Outcome column exists:", "Units Sold" in df.columns)

Total missing values: 0
Treatment column exists: True
Outcome column exists: True


In [14]:
# Check price variation across products and stores

price_variation = (
    df.groupby(["Store ID", "Product ID"])["Price"]
    .agg(["count", "nunique", "min", "max", "std"])
)

print("Total store-product combinations:", len(price_variation))

print(
    "Combinations with changing prices:",
    (price_variation["nunique"] > 1).sum()
)

print(
    "Combinations with fixed prices:",
    (price_variation["nunique"] == 1).sum()
)

print("\nSample price variation:")
display(price_variation.head(10))

Total store-product combinations: 100
Combinations with changing prices: 100
Combinations with fixed prices: 0

Sample price variation:


count  nunique    min     max        std
Store ID Product ID                                          
S001     P0001         760      681  43.36   90.74   8.832447
         P0002         760      684  37.73   99.74   9.891345
         P0003         760      649  35.11   81.12   7.616094
         P0004         760      709  64.91  142.17  13.987488
         P0005         760      690  36.98   90.20   8.851224
         P0006         760      643  22.47   57.41   6.037307
         P0007         760      592  14.92   37.09   3.774378
         P0008         760      640  24.47   64.22   5.716501
         P0009         760      689  49.07  167.34  12.802284
         P0010         760      707  80.83  177.46  15.168571

In [15]:
print("Total store-product combinations:", len(price_variation))

print(
    "Combinations with changing prices:",
    (price_variation["nunique"] > 1).sum()
)

print(
    "Combinations with fixed prices:",
    (price_variation["nunique"] == 1).sum()
)

print(
    "Average unique prices per combination:",
    round(price_variation["nunique"].mean(), 2)
)

Total store-product combinations: 100
Combinations with changing prices: 100
Combinations with fixed prices: 0
Average unique prices per combination: 653.86


In [16]:
# Examine the treatment variable: Price

print("Price distribution:")
display(df["Price"].describe().round(2))

print("\nTotal unique prices:", df["Price"].nunique())

print("\nPrice quantiles:")
display(
    df["Price"].quantile(
        [0, 0.10, 0.25, 0.50, 0.75, 0.90, 1]
    ).round(2)
)

Price distribution:


count    76000.00
mean        67.73
std         39.38
min          4.74
25%         32.00
50%         64.50
75%         95.83
max        228.03
Name: Price, dtype: float64


Total unique prices: 15396

Price quantiles:


0.00      4.74
0.10     19.86
0.25     32.00
0.50     64.50
0.75     95.83
0.90    121.92
1.00    228.03
Name: Price, dtype: float64

## 2. Dataset Source and Limitations

### Dataset Source
- Platform: Kaggle
- Dataset: Retail sales and inventory data
- File used: sales_data.csv
- Records: 76,000
- Features: 18

### Research Variables
- Treatment: Price
- Outcome: Units Sold

### Data Limitations
- Dataset provenance and exact original file version require confirmation.
- The dataset may be synthetic.
- Observational relationships do not establish causation.
- Potential confounders must be evaluated before DML estimation.
- Variable timing must be verified to avoid post-treatment bias.

## 3. Investigating Potential Confounders

We examine whether discounts, promotions, and competitor pricing are associated with the product price and sales outcome.

These relationships are descriptive and do not establish that the variables are valid causal controls.

In [17]:
# Compare price and sales across promotion status

promotion_analysis = (
    df.groupby("Promotion")
    .agg(
        Average_Price=("Price", "mean"),
        Average_Units_Sold=("Units Sold", "mean"),
        Record_Count=("Price", "count")
    )
    .round(2)
)

display(promotion_analysis)

# Compare price and sales across discount levels

discount_analysis = (
    df.groupby("Discount")
    .agg(
        Average_Price=("Price", "mean"),
        Average_Units_Sold=("Units Sold", "mean"),
        Record_Count=("Price", "count")
    )
    .round(2)
)

display(discount_analysis)

,Average_Price,Average_Units_Sold,Record_Count
Promotion,,,
0,68.92,81.83,51000
1,65.30,103.10,25000


,Average_Price,Average_Units_Sold,Record_Count
Discount,,,
0,73.07,81.57,17126
5,68.53,81.69,16918
10,66.93,87.66,23298
15,67.08,102.88,6245
20,63.45,103.77,6191
25,58.74,103.59,6222


In [18]:
# Relationship between our price and competitor pricing

print("Price vs Competitor Pricing correlation:")

correlation = df[
    ["Price", "Competitor Pricing"]
].corr()

display(correlation.round(3))

# Summary statistics
display(
    df[["Price", "Competitor Pricing"]]
    .describe()
    .round(2)
)

Price vs Competitor Pricing correlation:


,Price,Competitor Pricing
Price,1.000,0.977
Competitor Pricing,0.977,1.000


,Price,Competitor Pricing
count,76000.00,76000.00
mean,67.73,69.45
std,39.38,40.94
min,4.74,4.29
25%,32.00,32.62
50%,64.50,65.70
75%,95.83,97.93
max,228.03,261.22


In [19]:
# Analyze promotion and competitor pricing

promotion_competitor = (
    df.groupby("Promotion")
    .agg(
        Average_Price=("Price", "mean"),
        Average_Competitor_Price=("Competitor Pricing", "mean"),
        Average_Units_Sold=("Units Sold", "mean"),
        Records=("Price", "count")
    )
    .round(2)
)

display(promotion_competitor)

,Average_Price,Average_Competitor_Price,Average_Units_Sold,Records
Promotion,,,,
0,68.92,70.64,81.83,51000
1,65.30,67.03,103.10,25000


## 4. Preliminary Findings: Pricing and Promotions

### Discount Analysis
- Higher discount levels were generally associated with higher average sales.
- Average selling prices tended to decrease as discounts increased.

### Promotion Analysis
- Records without promotion: 51,000
- Records with promotion: 25,000
- Average sales without promotion: 81.83 units
- Average sales with promotion: 103.10 units
- Average price without promotion: 68.92
- Average price with promotion: 65.30

### Competitor Pricing
- Correlation between Price and Competitor Pricing: 0.977.
- This indicates a very strong positive linear association.

### Important Limitation
These findings are descriptive. The timing and causal role of potential control variables must be established before DML estimation.

## 5. Preliminary DML Variable Strategy

### Treatment
Price

### Outcome
Units Sold

### Potential Control Variables
Store ID, Product ID, Date, Promotion, Competitor Pricing,
Weather Condition, Seasonality, and Region.

### Variables Requiring Further Investigation
Discount, Category, and Epidemic.

### Variables Excluded Initially
Demand, Inventory Level, and Units Ordered.

The final adjustment set will depend on variable definitions,
measurement timing, and causal assumptions.

## 4. Preliminary Findings: Pricing and Promotions

### Discount Analysis
- Higher discount levels were generally associated with higher average sales.
- Average selling prices tended to decrease as discounts increased.

### Promotion Analysis
- Records without promotion: 51,000
- Records with promotion: 25,000
- Average sales without promotion: 81.83 units
- Average sales with promotion: 103.10 units
- Average price without promotion: 68.92
- Average price with promotion: 65.30

### Competitor Pricing
- Correlation between Price and Competitor Pricing: 0.977.
- This indicates a very strong positive linear association.

### Important Limitation
These findings are descriptive. The timing and causal role of potential control variables must be established before DML estimation.

In [20]:
# Inspect data types of proposed DML variables

candidate_columns = [
    "Price",
    "Units Sold",
    "Store ID",
    "Product ID",
    "Date",
    "Promotion",
    "Competitor Pricing",
    "Weather Condition",
    "Seasonality",
    "Region",
    "Discount",
    "Category",
    "Epidemic"
]

variable_summary = pd.DataFrame({
    "Column": candidate_columns,
    "Data Type": [df[col].dtype for col in candidate_columns],
    "Unique Values": [df[col].nunique() for col in candidate_columns]
})

display(variable_summary)

,Column,Data Type,Unique Values
0,Price,float64,15396
1,Units Sold,int64,330
2,Store ID,str,5
3,Product ID,str,20
4,Date,datetime64[us],760
5,Promotion,int64,2
6,Competitor Pricing,float64,15963
7,Weather Condition,str,4
8,Seasonality,str,4
9,Region,str,4


In [21]:
# Examine dataset time coverage

print("First date:", df["Date"].min())
print("Last date:", df["Date"].max())

print("Number of unique dates:", df["Date"].nunique())

print("\nRecords per date:")
display(
    df.groupby("Date").size().describe().round(2)
)

First date: 2022-01-01 00:00:00
Last date: 2024-01-30 00:00:00
Number of unique dates: 760

Records per date:


count    760.0
mean     100.0
std        0.0
min      100.0
25%      100.0
50%      100.0
75%      100.0
max      100.0
dtype: float64

In [22]:
print("First date:", df["Date"].min())
print("Last date:", df["Date"].max())
print("Unique dates:", df["Date"].nunique())
print("Records per date:", df.groupby("Date").size().mean())

First date: 2022-01-01 00:00:00
Last date: 2024-01-30 00:00:00
Unique dates: 760
Records per date: 100.0



## 7. Preparing Data for Double Machine Learning

### Objective
Prepare the treatment, outcome, and potential confounding variables for the DML estimation process.

### Treatment (T)
Price

### Outcome (Y)
Units Sold

### Potential Controls (X)
- Store ID
- Product ID
- Date
- Weather Condition
- Seasonality
- Epidemic

### Data Preparation Checks
- Verify data types.
- Check for missing values.
- Check treatment variation.
- Examine treatment overlap across relevant groups.
- Confirm that control variables are measured before treatment.

### Important Limitation
The selected controls are preliminary. Their inclusion in the final DML model depends on their causal relevance and measurement timing.


In [3]:

import pandas as pd
from pathlib import Path

# Automatically locate the EconoCausal project root
current_dir = Path.cwd()

project_dir = None

for folder in [current_dir, *current_dir.parents]:
    if (folder / "data" / "raw" / "sales_data.csv").exists():
        project_dir = folder
        break

if project_dir is None:
    raise FileNotFoundError(
        "Could not locate data/raw/sales_data.csv. "
        "Check that the dataset exists inside the EconoCausal project."
    )

# Load the dataset
file_path = project_dir / "data" / "raw" / "sales_data.csv"

df = pd.read_csv(file_path, parse_dates=["Date"])

print("Dataset loaded successfully!")
print("Project directory:", project_dir)
print("Dataset location:", file_path)
print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())


Dataset loaded successfully!
Project directory: c:\Users\methu\Desktop\EconoCausal
Dataset location: c:\Users\methu\Desktop\EconoCausal\data\raw\sales_data.csv
Dataset shape: (76000, 16)

Columns:
['Date', 'Store ID', 'Product ID', 'Category', 'Region', 'Inventory Level', 'Units Sold', 'Units Ordered', 'Price', 'Discount', 'Weather Condition', 'Promotion', 'Competitor Pricing', 'Seasonality', 'Epidemic', 'Demand']


In [4]:

# 1. Define the outcome variable (Y)
Y = df["Units Sold"].astype(float)

# 2. Define the treatment variable (T)
T = df["Price"].astype(float)

# 3. Select preliminary potential control variables (X)
X = df[
    [
        "Store ID",
        "Product ID",
        "Date",
        "Weather Condition",
        "Seasonality",
        "Epidemic"
    ]
].copy()

# 4. Check dimensions
print("Outcome (Y):", Y.shape)
print("Treatment (T):", T.shape)
print("Potential controls (X):", X.shape)

# 5. Check missing values
print("\nMissing values:")
print("Y:", Y.isnull().sum())
print("T:", T.isnull().sum())
print("X:", X.isnull().sum().sum())

# 6. Display sample records
print("\nPotential controls:")
display(X.head())

print("\nTreatment and outcome:")
display(pd.DataFrame({
    "Price": T,
    "Units Sold": Y
}).head())


Outcome (Y): (76000,)
Treatment (T): (76000,)
Potential controls (X): (76000, 6)

Missing values:
Y: 0
T: 0
X: 0

Potential controls:


,Store ID,Product ID,Date,Weather Condition,Seasonality,Epidemic
0,S001,P0001,2022-01-01,Snowy,Winter,0
1,S001,P0002,2022-01-01,Snowy,Winter,0
2,S001,P0003,2022-01-01,Snowy,Winter,0
3,S001,P0004,2022-01-01,Snowy,Winter,0
4,S001,P0005,2022-01-01,Snowy,Winter,0



Treatment and outcome:


,Price,Units Sold
0,72.72,102.0
1,80.16,117.0
2,62.94,114.0
3,87.63,45.0
4,54.41,65.0


In [5]:

# Inspect data types of potential control variables
print("CONTROL VARIABLE DATA TYPES")
print(X.dtypes)

# Count unique values in each control
print("\nUNIQUE VALUE COUNTS")
print(X.nunique())

# Inspect categorical distributions
categorical_columns = [
    "Store ID",
    "Product ID",
    "Weather Condition",
    "Seasonality",
    "Epidemic"
]

print("\nCATEGORICAL VARIABLE DISTRIBUTIONS")

for col in categorical_columns:
    print(f"\n{col}:")
    print(X[col].value_counts().head(10))

# Inspect date coverage
print("\nDATE COVERAGE")
print("First date:", X["Date"].min())
print("Last date:", X["Date"].max())
print("Unique dates:", X["Date"].nunique())


CONTROL VARIABLE DATA TYPES
Store ID                        str
Product ID                      str
Date                 datetime64[us]
Weather Condition               str
Seasonality                     str
Epidemic                      int64
dtype: object

UNIQUE VALUE COUNTS
Store ID               5
Product ID            20
Date                 760
Weather Condition      4
Seasonality            4
Epidemic               2
dtype: int64

CATEGORICAL VARIABLE DISTRIBUTIONS

Store ID:
Store ID
S001    15200
S002    15200
S003    15200
S004    15200
S005    15200
Name: count, dtype: int64

Product ID:
Product ID
P0001    3800
P0002    3800
P0003    3800
P0004    3800
P0005    3800
P0006    3800
P0007    3800
P0008    3800
P0009    3800
P0010    3800
Name: count, dtype: int64

Weather Condition:
Weather Condition
Cloudy    24360
Sunny     22980
Rainy     17500
Snowy     11160
Name: count, dtype: int64

Seasonality:
Seasonality
Winter    21000
Spring    18400
Summer    18400
Autumn    1820

In [6]:

# Check price variation within each store-product combination

price_variation = (
    df.groupby(["Store ID", "Product ID"])["Price"]
    .agg(["count", "nunique", "min", "max", "std"])
)

print("STORE-PRODUCT PRICE VARIATION")
print("Total combinations:", len(price_variation))

print(
    "Combinations with changing prices:",
    (price_variation["nunique"] > 1).sum()
)

print(
    "Combinations with fixed prices:",
    (price_variation["nunique"] == 1).sum()
)

print("\nPRICE VARIATION SUMMARY")
display(price_variation.describe().round(2))

# Check overall treatment distribution
print("\nOVERALL PRICE DISTRIBUTION")
display(df["Price"].describe().round(2))


STORE-PRODUCT PRICE VARIATION
Total combinations: 100
Combinations with changing prices: 100
Combinations with fixed prices: 0

PRICE VARIATION SUMMARY


,count,nunique,min,max,std
count,100.0,100.00,100.00,100.00,100.00
mean,760.0,653.86,42.24,101.11,9.43
std,0.0,68.52,25.07,54.81,4.81
min,760.0,374.00,4.74,12.12,1.18
25%,760.0,630.25,19.65,48.38,5.19
50%,760.0,681.00,38.46,96.57,9.04
75%,760.0,704.00,58.63,140.26,13.70
max,760.0,720.00,104.48,228.03,20.34



OVERALL PRICE DISTRIBUTION


count    76000.00
mean        67.73
std         39.38
min          4.74
25%         32.00
50%         64.50
75%         95.83
max        228.03
Name: Price, dtype: float64

In [7]:

# Calculate price ranges for each store-product combination

price_support = (
    df.groupby(["Store ID", "Product ID"])["Price"]
    .agg(
        min_price="min",
        max_price="max",
        mean_price="mean",
        unique_prices="nunique"
    )
    .reset_index()
)

print("Number of store-product combinations:", len(price_support))

print("\nPRICE SUPPORT SUMMARY")
display(price_support.describe().round(2))

# Check the overall common price interval
common_min = price_support["min_price"].max()
common_max = price_support["max_price"].min()

print("\nCOMMON PRICE INTERVAL")
print("Lower boundary:", round(common_min, 2))
print("Upper boundary:", round(common_max, 2))

if common_min <= common_max:
    print("A common price interval exists across all combinations.")
else:
    print("No single common price interval exists across all combinations.")


Number of store-product combinations: 100

PRICE SUPPORT SUMMARY


,min_price,max_price,mean_price,unique_prices
count,100.00,100.00,100.00,100.00
mean,42.24,101.11,67.73,653.86
std,25.07,54.81,38.12,68.52
min,4.74,12.12,7.81,374.00
25%,19.65,48.38,32.08,630.25
50%,38.46,96.57,63.80,681.00
75%,58.63,140.26,95.03,704.00
max,104.48,228.03,155.98,720.00



COMMON PRICE INTERVAL
Lower boundary: 104.48
Upper boundary: 12.12
No single common price interval exists across all combinations.



### Treatment Overlap Investigation

The dataset contains 100 store-product combinations.

The highest minimum price across combinations is 104.48, while the lowest maximum price is 12.12.

Therefore, no single common price interval exists across all store-product combinations.

This indicates differences in price support between products and stores. However, this diagnostic alone does not establish a failure of the overlap assumption.

Further investigation of conditional treatment variation is required before fitting the DML estimator.



## 8. Preliminary Treatment Model

### Objective
Investigate how much variation in product price can be explained by the preliminary control variables.

### Method
A Random Forest Regressor will be used to predict Price from the selected controls.

The difference between actual and predicted Price represents residual treatment variation.

### Important Limitation
This is a preliminary diagnostic. The control variables and their timing must be validated before using the residuals in a causal estimate.


In [8]:

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.metrics import mean_absolute_error, r2_score

# Prepare categorical and date-based features
X_treatment = X.drop(columns=["Date"]).copy()

# Convert date into a simple time index for this diagnostic
X_treatment["Day_Index"] = (
    (X["Date"] - X["Date"].min()).dt.days
)

categorical_features = [
    "Store ID",
    "Product ID",
    "Weather Condition",
    "Seasonality"
]

numeric_features = ["Epidemic", "Day_Index"]

preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", OneHotEncoder(handle_unknown="ignore"),
         categorical_features),
        ("numeric", "passthrough", numeric_features)
    ]
)

treatment_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", RandomForestRegressor(
            n_estimators=50,
            max_depth=12,
            min_samples_leaf=5,
            random_state=42,
            n_jobs=-1
        ))
    ]
)

# Generate out-of-fold price predictions
cv = KFold(n_splits=5, shuffle=True, random_state=42)

T_pred = cross_val_predict(
    treatment_model,
    X_treatment,
    T,
    cv=cv,
    n_jobs=1
)

# Calculate treatment residuals
T_residual = T - T_pred

print("TREATMENT MODEL RESULTS")
print("MAE:", round(mean_absolute_error(T, T_pred), 3))
print("R-squared:", round(r2_score(T, T_pred), 3))

print("\nTREATMENT RESIDUAL SUMMARY")
print(pd.Series(T_residual).describe().round(3))


TREATMENT MODEL RESULTS
MAE: 16.315
R-squared: 0.678

TREATMENT RESIDUAL SUMMARY
count    76000.000
mean        -0.021
std         22.341
min        -65.821
25%        -12.358
50%         -1.142
75%          9.948
max        120.861
Name: Price, dtype: float64


In [9]:

print("TREATMENT MODEL PERFORMANCE")

print(
    "MAE:",
    round(mean_absolute_error(T, T_pred), 3)
)

print(
    "R-squared:",
    round(r2_score(T, T_pred), 3)
)

print("\nACTUAL VS PREDICTED PRICE")
display(pd.DataFrame({
    "Actual Price": T.head(10).values,
    "Predicted Price": T_pred[:10]
}).round(2))

print("\nCORRELATION")
print(
    "Actual Price vs Predicted Price:",
    round(np.corrcoef(T, T_pred)[0, 1], 3)
)


TREATMENT MODEL PERFORMANCE
MAE: 16.315
R-squared: 0.678

ACTUAL VS PREDICTED PRICE


,Actual Price,Predicted Price
0,72.72,56.60
1,80.16,56.60
2,62.94,56.11
3,87.63,56.60
4,54.41,56.65
5,35.53,56.96
6,23.27,25.34
7,37.79,56.80
8,94.20,93.79
9,113.35,56.60



CORRELATION
Actual Price vs Predicted Price: 0.823



## 9. Preliminary Treatment Model Findings

### Dataset
- Observations: 76,000
- Treatment: Price
- Outcome: Units Sold

### Treatment Model
A Random Forest Regressor was used to predict Price from preliminary control variables using five-fold shuffled cross-validation.

### Findings
- Actual and predicted price correlation: 0.823.
- Treatment residuals have a standard deviation of 22.341.
- Residual mean is approximately zero (-0.021).
- No single common price interval was found across all store-product combinations.

### Interpretation
The preliminary controls explain some variation in product prices, while substantial residual variation remains.

These results are diagnostic and do not establish causal identification.

### Next Steps
- Review the causal validity and timing of control variables.
- Improve the validation strategy to account for time and repeated store-product observations.
- Evaluate the outcome model.
- Implement and validate the final DML estimator.




## 10. Time-Aware Treatment Model Validation

### Objective
Evaluate treatment prediction using chronological validation rather than randomly shuffled observations.

### Method
Use TimeSeriesSplit with five folds based on unique dates.

Each validation fold will contain dates later than its training data.

### Reason
Randomly shuffled validation can allow future observations to influence predictions for earlier dates.

### Limitation
This is a more appropriate temporal diagnostic, but it does not by itself establish causal identification or solve repeated-observation dependence.


In [2]:

import pandas as pd
import numpy as np

from pathlib import Path
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, r2_score

# Locate project root
current_dir = Path.cwd()

project_dir = next(
    (
        folder for folder in [current_dir, *current_dir.parents]
        if (folder / "data" / "raw" / "sales_data.csv").exists()
    ),
    None
)

if project_dir is None:
    raise FileNotFoundError("Could not locate sales_data.csv")

# Load dataset
df = pd.read_csv(
    project_dir / "data" / "raw" / "sales_data.csv",
    parse_dates=["Date"]
)

# Recreate treatment and potential controls
T = df["Price"].astype(float)

X = df[
    [
        "Store ID",
        "Product ID",
        "Date",
        "Weather Condition",
        "Seasonality",
        "Epidemic"
    ]
].copy()

# Prepare treatment model features
X_treatment = X.drop(columns=["Date"]).copy()
X_treatment["Day_Index"] = (
    X["Date"] - X["Date"].min()
).dt.days

categorical_features = [
    "Store ID",
    "Product ID",
    "Weather Condition",
    "Seasonality"
]

numeric_features = ["Epidemic", "Day_Index"]

# Recreate treatment model
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor

preprocessor = ColumnTransformer([
    (
        "categorical",
        OneHotEncoder(handle_unknown="ignore"),
        categorical_features
    ),
    ("numeric", "passthrough", numeric_features)
])

treatment_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=50,
        max_depth=12,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1
    ))
])

# Chronological validation
unique_dates = np.sort(X["Date"].unique())
tscv = TimeSeriesSplit(n_splits=5)

T_pred_time = np.full(len(T), np.nan)

for fold, (train_idx, test_idx) in enumerate(
    tscv.split(unique_dates), start=1
):
    train_dates = unique_dates[train_idx]
    test_dates = unique_dates[test_idx]

    train_mask = X["Date"].isin(train_dates)
    test_mask = X["Date"].isin(test_dates)

    treatment_model.fit(
        X_treatment.loc[train_mask],
        T.loc[train_mask]
    )

    T_pred_time[test_mask] = treatment_model.predict(
        X_treatment.loc[test_mask]
    )

    print(
        f"Fold {fold}: "
        f"Train={train_mask.sum()}, "
        f"Test={test_mask.sum()}"
    )

# Evaluate predictions
valid = ~np.isnan(T_pred_time)

print("\nTIME-AWARE TREATMENT MODEL")
print("Evaluated rows:", valid.sum())
print(
    "MAE:",
    round(mean_absolute_error(T[valid], T_pred_time[valid]), 3)
)
print(
    "R-squared:",
    round(r2_score(T[valid], T_pred_time[valid]), 3)
)

T_residual_time = T[valid].to_numpy() - T_pred_time[valid]

print(
    "Residual standard deviation:",
    round(np.std(T_residual_time), 3)
)


Fold 1: Train=13000, Test=12600
Fold 2: Train=25600, Test=12600
Fold 3: Train=38200, Test=12600
Fold 4: Train=50800, Test=12600
Fold 5: Train=63400, Test=12600

TIME-AWARE TREATMENT MODEL
Evaluated rows: 63000
MAE: 16.448
R-squared: 0.677
Residual standard deviation: 22.34



### Time-Aware Treatment Model Results

A Random Forest Regressor was evaluated using five chronological validation folds.

**Results:**
- Evaluated observations: 63,000
- MAE: 16.448
- R-squared: 0.677
- Treatment residual standard deviation: 22.340

**Interpretation:**

The model achieved an R-squared of 0.677 when predicting prices on later dates using earlier observations.

The remaining residual variation indicates that the selected controls do not fully explain price variation.

**Limitation:**

This is a treatment prediction diagnostic, not a causal estimate. Further validation is needed to account for repeated store-product observations and to establish a defensible DML cross-fitting strategy.



## 11. Preliminary Outcome Model

### Objective
Investigate how well the preliminary control variables predict Units Sold.

### Outcome Variable (Y)
Units Sold

### Method
A Random Forest Regressor will be used to predict Units Sold using the preliminary control variables.

### Evaluation
- Mean Absolute Error (MAE)
- R-squared
- Out-of-time prediction performance

### Important Limitation
This is a predictive diagnostic, not a causal estimate. The final outcome model must use a defensible control set and an appropriate cross-fitting strategy.


In [5]:

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, r2_score

# 1. Prepare data directly from the main dataset
data = df.copy().reset_index(drop=True)
data["Date"] = pd.to_datetime(data["Date"])

target = "Units Sold"

categorical_cols = [
    "Store ID",
    "Product ID",
    "Weather Condition",
    "Seasonality"
]

numeric_cols = ["Epidemic"]

feature_cols = categorical_cols + numeric_cols

# Check required columns
required_cols = ["Date", target] + feature_cols
missing_cols = [c for c in required_cols if c not in data.columns]

if missing_cols:
    raise ValueError(f"Missing columns: {missing_cols}")

X = data[feature_cols].copy()
Y = data[target].astype(float)
dates = data["Date"]

# 2. Encode categorical columns
preprocessor = ColumnTransformer([
    ("categorical",
     OneHotEncoder(handle_unknown="ignore"),
     categorical_cols),
    ("numerical", "passthrough", numeric_cols)
])

model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=50,
        max_depth=12,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1
    ))
])

# 3. Five-fold chronological validation
unique_dates = np.sort(dates.unique())
splitter = TimeSeriesSplit(n_splits=5)

predictions = np.full(len(data), np.nan)

for fold, (train_dates_idx, test_dates_idx) in enumerate(
    splitter.split(unique_dates), start=1
):
    train_dates = unique_dates[train_dates_idx]
    test_dates = unique_dates[test_dates_idx]

    train_mask = dates.isin(train_dates).to_numpy()
    test_mask = dates.isin(test_dates).to_numpy()

    model.fit(X.loc[train_mask], Y.loc[train_mask])
    predictions[test_mask] = model.predict(X.loc[test_mask])

    print(
        f"Fold {fold}: "
        f"Train={train_mask.sum()}, "
        f"Test={test_mask.sum()}"
    )

# 4. Evaluate
valid = ~np.isnan(predictions)

print("\nOUTCOME MODEL RESULTS")
print("Evaluated rows:", valid.sum())
print("MAE:", round(mean_absolute_error(Y[valid], predictions[valid]), 3))
print("R-squared:", round(r2_score(Y[valid], predictions[valid]), 3))

# 5. Calculate residuals
Y_residual_time = Y.to_numpy()[valid] - predictions[valid]

print(
    "Residual standard deviation:",
    round(np.std(Y_residual_time), 3)
)


Fold 1: Train=13000, Test=12600
Fold 2: Train=25600, Test=12600
Fold 3: Train=38200, Test=12600
Fold 4: Train=50800, Test=12600
Fold 5: Train=63400, Test=12600

OUTCOME MODEL RESULTS
Evaluated rows: 63000
MAE: 29.566
R-squared: 0.227
Residual standard deviation: 38.788



## Outcome Model — Time-Aware Validation

A Random Forest Regressor was used to predict Units Sold using store, product, weather, seasonality, and epidemic information.

### Results
- Evaluated observations: 63,000
- MAE: 29.566
- R-squared: 0.227
- Residual standard deviation: 38.788

### Interpretation
The model explains approximately 22.7% of the variation in Units Sold on later dates. Its predictive performance is limited, indicating that the selected variables do not capture all factors associated with sales.

### Limitation
These results evaluate outcome prediction only. They do not establish a causal relationship between Price and Units Sold.
